# 02 — Understat Data Collection

Collects xG/xA data from Understat and saves raw files.
Then fuzzy-matches Understat player names to FPL player names
and builds a unified dataset.

**Outputs:**
- `data/raw/understat_players.parquet` — season totals per player
- `data/raw/understat_matches.parquet` — match-by-match xG/xA
- `data/processed/player_id_map.parquet` — FPL ↔ Understat ID mapping
- `data/processed/merged_players.parquet` — unified dataset for modelling

## 0. Smoke test — verify Understat endpoints are reachable

In [1]:
import asyncio
import sys
sys.path.insert(0, '..')

from src.data.understat_client import UnderstatClient

async def smoke_test():
    async with UnderstatClient() as client:
        players = await client.get_league_players(season='2025')
        print(f'Season totals: {len(players)} players')
        print(f'Columns: {list(players.columns)}')
        print(players.head(3).to_string(index=False))

await smoke_test()

Season totals: 521 players
Columns: ['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'yellow_cards', 'red_cards', 'position', 'team_title']
   id     player_name games time goals        xG assists       xA shots key_passes npg      npxG   xGChain  xGBuildup yellow_cards red_cards position                  team_title
 8260  Erling Haaland    29 2439    22 23.136934       7 4.766327   102         21  19 20.092258 26.238575   4.129314            1         0      F S             Manchester City
13222          Thiago    31 2662    19 21.101610       1 3.155416    71         19  13 15.773428 19.475172   3.493408            6         0      F S                   Brentford
11363 Antoine Semenyo    29 2597    15 11.201725       4 3.091043    68         32  14  9.679387 16.244874   4.950853            6         0      F M Bournemouth,Manchester City


## 1. Imports & setup

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

## 2. Fetch season totals

In [3]:
async def fetch_season_totals():
    async with UnderstatClient() as client:
        return await client.get_league_players(season='2025')

understat_players = await fetch_season_totals()
print(understat_players.shape)
understat_players.head()

(521, 18)


,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,npg,npxG,xGChain,xGBuildup,yellow_cards,red_cards,position,team_title
0,8260,Erling Haaland,29,2439,22,23.136934,7,4.766327,102,21,19,20.092258,26.238575,4.129314,1,0,F S,Manchester City
1,13222,Thiago,31,2662,19,21.101610,1,3.155416,71,19,13,15.773428,19.475172,3.493408,6,0,F S,Brentford
2,11363,Antoine Semenyo,29,2597,15,11.201725,4,3.091043,68,32,14,9.679387,16.244874,4.950853,6,0,F M,"Bournemouth,Manchester City"
3,8272,João Pedro,31,2345,14,14.158351,5,3.960961,63,28,14,14.158351,17.678359,4.659978,4,0,F M S,Chelsea
4,501,Danny Welbeck,30,1832,12,10.899193,0,1.876528,43,18,11,8.615687,12.241746,3.454270,5,0,F S,Brighton


## 3. Fetch match-by-match data for all players

Uses 10 concurrent workers with 100ms delay. ~600 players → ~2-3 minutes.

In [4]:
async def fetch_all_matches(player_ids):
    async with UnderstatClient(concurrency=10, request_delay=0.1) as client:
        return await client.get_all_player_matches(
            player_ids,
            season_filter='2025',
        )

player_ids = understat_players['id'].tolist()
understat_matches = await fetch_all_matches(player_ids)

print(f'Match rows: {len(understat_matches)}')
print(f'Unique players: {understat_matches["understat_id"].nunique()}')
understat_matches.head()

Match rows: 9642
Unique players: 521


,understat_id,date,season,h_team,a_team,h_goals,a_goals,goals,xG,assists,xA,shots,key_passes,npg,npxG,xGChain,xGBuildup,time,position
0,8875,2026-03-16,2025,Brentford,Wolverhampton Wanderers,2,2,1,0.836452,0,0.000000,2,0,1,0.836452,0.384336,0.030594,13,Sub
1,8875,2026-03-03,2025,Wolverhampton Wanderers,Liverpool,2,1,0,0.000000,1,0.392227,0,1,0,0.000000,0.392227,0.000000,28,Sub
2,8875,2026-02-27,2025,Wolverhampton Wanderers,Aston Villa,2,0,0,0.000000,0,0.073271,0,1,0,0.000000,0.073271,0.000000,8,Sub
3,8875,2026-02-22,2025,Crystal Palace,Wolverhampton Wanderers,1,0,0,1.358522,0,0.000000,5,0,0,0.597353,0.081317,0.000000,90,FW
4,8875,2026-02-18,2025,Wolverhampton Wanderers,Arsenal,2,2,0,0.009980,0,0.000000,1,0,0,0.009980,0.009980,0.009980,69,Sub


In [5]:
# Basic sanity checks
print('Date range:', understat_matches['date'].min(), '→', understat_matches['date'].max())
print('Seasons present:', understat_matches['season'].unique())
print('Null xG:', understat_matches['xG'].isna().sum())
print('Null xA:', understat_matches['xA'].isna().sum())

Date range: 2025-08-15 00:00:00 → 2026-04-04 00:00:00
Seasons present: <ArrowStringArray>
['2025']
Length: 1, dtype: str
Null xG: 0
Null xA: 0


## 4. Save raw Understat files

In [6]:
understat_players.to_parquet(RAW / 'understat_players.parquet', index=False)
understat_matches.to_parquet(RAW / 'understat_matches.parquet', index=False)

print('Saved:')
print(f'  understat_players.parquet  — {len(understat_players)} rows')
print(f'  understat_matches.parquet  — {len(understat_matches)} rows')

Saved:
  understat_players.parquet  — 521 rows
  understat_matches.parquet  — 9642 rows


## 5. Fuzzy name matching — FPL ↔ Understat

The core challenge: FPL uses short names (`"Salah"`, `"Trent"`) while Understat
uses full names (`"Mohamed Salah"`, `"Trent Alexander-Arnold"`). We can't join on
exact string match — we need fuzzy matching.

**Strategy:**
- Build a full name for each FPL player from `first_name + second_name`
- Use `rapidfuzz.process.extractOne` with `token_sort_ratio` (handles word-order differences)
- Score ≥ 85 → auto-match
- Score 70–84 → flag for manual review
- Manual override dict for known hard cases

In [7]:
from rapidfuzz import process, fuzz

# Load FPL players
fpl_players = pd.read_parquet(RAW / 'fpl_players.parquet')
fpl_players['full_name'] = fpl_players['first_name'] + ' ' + fpl_players['second_name']

print('FPL players:', len(fpl_players))
print('Understat players:', len(understat_players))

FPL players: 825
Understat players: 521


In [ ]:
import unicodedata
from rapidfuzz import process, fuzz

# ── Name normalisation ────────────────────────────────────────────────────────
def normalize_name(name: str) -> str:
    """Lowercase, remove accents, replace hyphens/apostrophes with spaces."""
    if not isinstance(name, str):
        return ''
    # Decode HTML entities (e.g. &#039; → ')
    import html
    name = html.unescape(name)
    # Strip accents
    name = unicodedata.normalize('NFD', name)
    name = ''.join(c for c in name if unicodedata.category(c) != 'Mn')
    # Hyphens and apostrophes → space
    name = name.replace('-', ' ').replace("'", ' ')
    return name.lower().strip()

# ── Manual overrides for genuinely hard cases ─────────────────────────────────
# Format: {understat_name: fpl_full_name}
MANUAL_OVERRIDES = {
    # One-name players whose FPL name includes surname
    'Alisson':              'Alisson Becker',
    'Casemiro':             'Carlos Henrique Casimiro',
    'Richarlison':          'Richarlison de Andrade',
    'Evanilson':            'Francisco Evanilson de Lima Barbosa',
    'Joelinton':            'Joelinton Cassio Apolinario de Lira',
    'Estêvão':              'Estevao Almeida de Oliveira Goncalves',
    'Rodri':                'Rodrigo Hernandez Cascante',
    'Gabriel':              'Gabriel dos Santos Magalhaes',
    'Thiago':               'Thiago Silva',
    'Murillo':              'Murillo Costa dos Santos',
    'Reinildo':             'Reinildo Mandava',
    'Toti':                 'Toti Gomes',
    'Rayan':                'Rayan Vito Simplicio Rocha',
    'André':                'Andre Trindade da Costa Neto',
    'André':                'Andre Trindade da Costa Neto',
    # Nicknames vs legal names
    'Ben White':            'Benjamin White',
    'Matthew Cash':         'Matty Cash',
    'Valentino Livramento': 'Tino Livramento',
    'Max Kilman':           'Maximilian Kilman',
    'Bruno Fernandes':      'Bruno Borges Fernandes',
    'Destiny Udogie':       'Iyenoma Udogie',
    'Amad Diallo Traore':   'Amad Diallo',
    'Pape Sarr':            'Pape Matar Sarr',
    'Tomas Soucek':         'Tomas Soucek',
    'Pedro Neto':           'Pedro Lomba Neto',
    'Diogo Dalot':          'Diogo Dalot Teixeira',
    'Alejandro Garnacho':   'Alejandro Garnacho Ferreyra',
    'Rodrigo Muniz':        'Rodrigo Muniz Carvalho',
    # HTML entity names
    "Jun'ai Byfield":       "Jun'ai Byfield",
    "Nico O'Reilly":        'Nico O Reilly',
    "Matt O'Riley":         'Matt O Riley',
    "Jake O'Brien":         'Jake O Brien',
    "Luke O'Nien":          'Luke O Nien',
}

# Build FPL lookup with normalised keys
fpl_players_match = pd.read_parquet(RAW / 'fpl_players.parquet')
fpl_players_match['full_name'] = fpl_players_match['first_name'] + ' ' + fpl_players_match['second_name']
fpl_names_raw = fpl_players_match['full_name'].tolist()
fpl_name_to_id = dict(zip(fpl_players_match['full_name'], fpl_players_match['id']))
fpl_names_norm = [normalize_name(n) for n in fpl_names_raw]
fpl_norm_to_raw = dict(zip(fpl_names_norm, fpl_names_raw))

def best_match(u_name: str):
    """
    Return (fpl_raw_name, fpl_id, score) using a two-scorer strategy:
    1. token_set_ratio on normalised names (handles subsets like 'Alisson' ↔ 'Alisson Becker')
    2. partial_ratio as a fallback
    Take the higher of the two scores.
    """
    u_norm = normalize_name(u_name)
    m_set  = process.extractOne(u_norm, fpl_names_norm, scorer=fuzz.token_set_ratio)
    m_part = process.extractOne(u_norm, fpl_names_norm, scorer=fuzz.partial_ratio)
    if m_set[1] >= m_part[1]:
        best_norm, score = m_set[0], m_set[1]
    else:
        best_norm, score = m_part[0], m_part[1]
    raw_name = fpl_norm_to_raw.get(best_norm, best_norm)
    fpl_id   = fpl_name_to_id.get(raw_name)
    return raw_name, fpl_id, score

# ── Run matching ──────────────────────────────────────────────────────────────
records = []
for _, row in understat_players.iterrows():
    u_name = row['player_name']
    u_id   = int(row['id'])

    if u_name in MANUAL_OVERRIDES:
        fpl_name = MANUAL_OVERRIDES[u_name]
        fpl_id   = fpl_name_to_id.get(fpl_name)
        score    = 100
        status   = 'manual'
    else:
        fpl_name, fpl_id, score = best_match(u_name)
        if score >= 85:
            status = 'auto'
        elif score >= 70:
            status = 'review'
        else:
            status = 'no_match'

    records.append({
        'understat_id':   u_id,
        'understat_name': u_name,
        'fpl_id':         fpl_id,
        'fpl_name':       fpl_name,
        'match_score':    score,
        'match_status':   status,
    })

id_map = pd.DataFrame(records)
print('=== Matching results ===')
print(id_map['match_status'].value_counts())
print(f'\nTotal confirmed (auto + manual): {(id_map["match_status"].isin(["auto","manual"])).sum()}')
print(f'Review (check manually):         {(id_map["match_status"] == "review").sum()}')
print(f'No match (likely non-FPL):       {(id_map["match_status"] == "no_match").sum()}')

In [9]:
# Inspect players flagged for review
review = id_map[id_map['match_status'] == 'review']
print(f'{len(review)} players need review:')
review[['understat_name', 'fpl_name', 'match_score']].sort_values('match_score', ascending=False)

57 players need review:


,understat_name,fpl_name,match_score
287,Mateo Kovacic,Mateo Kovačić,84.615385
372,Radu Dragusin,Radu Drăgușin,84.615385
44,Emiliano Buendía,Emiliano Buendía Stati,84.210526
428,Marc Guiu,Marc Guéhi,84.210526
106,Idrissa Gueye,Idrissa Gana Gueye,83.870968
286,Jefferson Lerma,Jefferson Lerma Solís,83.333333
475,Jamaldeen Jimoh,Jamaldeen Jimoh-Aloba,83.333333
350,Arnaud Kalimuendo Muinga,Arnaud Kalimuendo,82.926829
424,Carlos Alcaraz,Carlos Alcaraz Durán,82.352941
371,Robert Sánchez,Robert Lynch Sánchez,82.352941


In [10]:
# Inspect no-matches (likely non-EPL players in Understat who aren't in FPL)
no_match = id_map[id_map['match_status'] == 'no_match']
print(f'{len(no_match)} players with no FPL match (expected — Understat includes players not in FPL):')
no_match[['understat_name', 'fpl_name', 'match_score']].head(20)

52 players with no FPL match (expected — Understat includes players not in FPL):


,understat_name,fpl_name,match_score
1,Thiago,Malick Thiaw,44.444444
3,João Pedro,Pedro Lomba Neto,61.538462
10,Richarlison,Richarlison de Andrade,66.666667
13,Bruno Guimarães,Bruno Guimarães Rodriguez Moura,65.217391
24,Casemiro,Tom Cairney,52.631579
27,Matheus Cunha,Mateus Mané,66.666667
38,Beto,Luka Bentt,42.857143
41,Evanilson,Evan Ferguson,63.636364
60,Lucas Paquetá,Luca Netz,63.636364
78,Gabriel,Gabriel Słonina,63.636364


In [ ]:
# Keep auto + manual matches for the merge; drop review and no_match
id_map_confirmed = id_map[id_map['match_status'].isin(['auto', 'manual'])].copy()
print(f'Confirmed matches (auto + manual): {len(id_map_confirmed)}')
print(f'Dropped (review + no_match):       {len(id_map) - len(id_map_confirmed)}')

id_map.to_parquet(PROCESSED / 'player_id_map.parquet', index=False)
print('Saved player_id_map.parquet')

## 6. Assign gameweek numbers to Understat matches

FPL data is indexed by `round` (gameweek 1–38). Understat data has match `date`.
We assign each Understat match to an FPL gameweek by finding which gameweek
deadline the match date falls between.

In [12]:
from src.data.fpl_client import FPLClient

fpl = FPLClient()
events = fpl.get_events()
events['deadline_time'] = pd.to_datetime(events['deadline_time'], utc=True)

# Build deadline lookup: gw → deadline
gw_deadlines = events[['id', 'deadline_time']].sort_values('id').reset_index(drop=True)
gw_deadlines.columns = ['round', 'deadline']
gw_deadlines.head()

,round,deadline
0,1,2025-08-15 17:30:00+00:00
1,2,2025-08-22 17:30:00+00:00
2,3,2025-08-30 10:00:00+00:00
3,4,2025-09-13 10:00:00+00:00
4,5,2025-09-20 10:00:00+00:00


In [13]:
def assign_gameweek(match_date, deadlines_df):
    """
    Return the FPL gameweek number for a given match date.
    A match belongs to gameweek N if it falls after the GW N deadline
    and before the GW N+1 deadline.
    """
    if pd.isna(match_date):
        return None
    match_date = match_date.tz_localize('UTC') if match_date.tzinfo is None else match_date
    mask = deadlines_df['deadline'] <= match_date
    eligible = deadlines_df[mask]
    if eligible.empty:
        return None
    return int(eligible.iloc[-1]['round'])

understat_matches['round'] = understat_matches['date'].apply(
    lambda d: assign_gameweek(d, gw_deadlines)
)

print('Gameweek assignment coverage:')
print(f'  Assigned: {understat_matches["round"].notna().sum()}')
print(f'  Unassigned: {understat_matches["round"].isna().sum()}')
understat_matches[['date', 'round', 'h_team', 'a_team', 'xG', 'xA']].head()

Gameweek assignment coverage:
  Assigned: 9608
  Unassigned: 34


,date,round,h_team,a_team,xG,xA
0,2026-03-16,30.0,Brentford,Wolverhampton Wanderers,0.836452,0.000000
1,2026-03-03,28.0,Wolverhampton Wanderers,Liverpool,0.000000,0.392227
2,2026-02-27,27.0,Wolverhampton Wanderers,Aston Villa,0.000000,0.073271
3,2026-02-22,27.0,Crystal Palace,Wolverhampton Wanderers,1.358522,0.000000
4,2026-02-18,26.0,Wolverhampton Wanderers,Arsenal,0.009980,0.000000


## 7. Merge FPL gameweeks + Understat matches

In [17]:
# Load FPL gameweek data
fpl_gw = pd.read_parquet(RAW / 'fpl_gameweeks.parquet')
print('FPL gameweek rows:', len(fpl_gw))

# Add understat_id to FPL data via id_map
fpl_gw = fpl_gw.merge(
    id_map_confirmed[['fpl_id', 'understat_id']],
    left_on='player_id',
    right_on='fpl_id',
    how='left',
).drop(columns=['fpl_id'])

print(f'Players with understat_id: {fpl_gw["understat_id"].notna().sum()} _/ {len(fpl_gw)}')

FPL gameweek rows: 23829
Players with understat_id: 12170 _/ 23829


In [18]:
# Aggregate Understat match data to one row per player per gameweek
# (some players have double gameweeks — sum xG/xA, take max minutes)
understat_agg = (
    understat_matches
    .dropna(subset=['round'])
    .groupby(['understat_id', 'round'])
    .agg(
        xG=('xG', 'sum'),
        xA=('xA', 'sum'),
        shots=('shots', 'sum'),
        key_passes=('key_passes', 'sum'),
        npxG=('npxG', 'sum'),
        xGChain=('xGChain', 'sum'),
        xGBuildup=('xGBuildup', 'sum'),
        us_minutes=('time', 'sum'),
    )
    .reset_index()
)

understat_agg['round'] = understat_agg['round'].astype(int)
print('Understat aggregated rows:', len(understat_agg))

Understat aggregated rows: 7630


In [19]:
# Merge on (understat_id, round)
merged = fpl_gw.merge(
    understat_agg,
    on=['understat_id', 'round'],
    how='left',
)

print('Merged shape:', merged.shape)
print('xG coverage:', f"{merged['xG'].notna().mean():.1%} of rows have Understat data")
merged.head()

Merged shape: (23829, 34)
xG coverage: 24.7% of rows have Understat data


,player_id,round,total_points,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,...,kickoff_time,understat_id,xG,xA,shots,key_passes,npxG,xGChain,xGBuildup,us_minutes
0,1,1,10,90,0,0,1,0,0,0,...,2025-08-17T15:30:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,6,90,0,0,1,0,0,0,...,2025-08-23T16:30:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,3,2,90,0,0,0,1,0,0,...,2025-08-31T15:30:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,4,6,90,0,0,1,0,0,0,...,2025-09-13T11:30:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,5,2,90,0,0,0,1,0,0,...,2025-09-21T15:30:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Add player metadata (position, team, name, price)
# position is derived at runtime by FPLClient — not stored in the raw parquet.
# Re-derive it here from element_type.
POSITION_MAP = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
fpl_players['position'] = fpl_players['element_type'].map(POSITION_MAP)

merged = merged.merge(
    fpl_players[['id', 'web_name', 'position', 'team', 'now_cost']],
    left_on='player_id',
    right_on='id',
    how='left',
).drop(columns=['id'])

print('Final shape:', merged.shape)
print('Columns:', list(merged.columns))

Final shape: (23829, 38)
Columns: ['player_id', 'round', 'total_points', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'value', 'selected', 'was_home', 'opponent_team', 'kickoff_time', 'understat_id', 'xG', 'xA', 'shots', 'key_passes', 'npxG', 'xGChain', 'xGBuildup', 'us_minutes', 'web_name', 'position', 'team', 'now_cost']


## 8. Save merged dataset

In [22]:
merged.to_parquet(PROCESSED / 'merged_players.parquet', index=False)

print('Saved merged_players.parquet')
print(f'  Rows: {len(merged)}')
print(f'  Columns: {len(merged.columns)}')
print(f'  Players: {merged["player_id"].nunique()}')
print(f'  Gameweeks: {merged["round"].nunique()}')
print(f'  xG coverage: {merged["xG"].notna().mean():.1%}')

Saved merged_players.parquet
  Rows: 23829
  Columns: 38
  Players: 825
  Gameweeks: 31
  xG coverage: 24.7%


In [23]:
# Coverage for players with >0 minutes
played = merged[merged['minutes'] > 0]
print(f"xG coverage for players who played: {played['xG'].notna().mean():.1%}")
print(f"Rows with minutes > 0: {len(played)}")


xG coverage for players who played: 56.7%
Rows with minutes > 0: 9345
